In [0]:
%sql
-- Creating a catalog and schema 
DROP table if exists cdc_catalog.cdc_schema.customer_source_table;
DROP table if exists cdc_catalog.cdc_schema.users_current;
DROP table if exists cdc_catalog.cdc_schema.users_history;
DROP table if exists cdc_catalog.cdc_schema.customers_cdf;
Create catalog if not exists cdc_catalog;
create schema if not exists cdc_catalog.cdc_schema;
create volume if not exists cdc_catalog.cdc_schema.cdc_customer_vol_ckpoint ;
GRANT ALL PRIVILEGES ON catalog  cdc_catalog TO `account users`;
GRANT ALL PRIVILEGES ON SCHEMA  cdc_catalog.cdc_schema TO `account users`;
use cdc_catalog.cdc_schema;

In [0]:
print('The below path will drop the checkout path...by default it will be disabled')
chkpnt_path = '/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint'
print(f"The checkpoint path :-------- {chkpnt_path}")
#dbutils.fs.rm(f"{chkpnt_path}",recurse=True)

The below path will drop the checkout path...by default it will be disabled
The checkpoint path :-------- /Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint


In [0]:
%sql
select current_schema(),current_catalog()

current_schema(),current_catalog()
cdc_schema,cdc_catalog


In [0]:
%sql
SELECT 
    grantee, 
    privilege_type, 
    'CATALOG' AS object_type, 
    catalog_name AS object_name
FROM information_schema.catalog_privileges
WHERE catalog_name = 'cdc_catalog'

UNION ALL

SELECT 
    grantee, 
    privilege_type, 
    'SCHEMA' AS object_type, 
    schema_name AS object_name
FROM information_schema.schema_privileges
WHERE catalog_name = 'cdc_catalog' 
  AND schema_name = 'cdc_schema';

grantee,privilege_type,object_type,object_name
account users,ALL_PRIVILEGES,CATALOG,cdc_catalog
account users,ALL_PRIVILEGES,SCHEMA,cdc_schema
account users,ALL_PRIVILEGES,SCHEMA,cdc_schema


In [0]:
%sql
Create table customer_source_table(
  userId  INT,name STRING , city STRING)
  USING DELTA 
  TBLPROPERTIES (delta.enableChangeDataFeed = true,
                 delta.deletedFileRetentionDuration = 'interval 1 days');

In [0]:
def calling_customer_cdf():
    spark.\
        readStream.\
            option("readChangeFeed", "true").\
                option("startingVersion", 0).\
                    table("customer_source_table").\
                        writeStream.\
                            option("checkpointLocation", "/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint").\
                                trigger (availableNow=True).\
                                    table("customers_cdf")

In [0]:
%sql
-----------Inserting data in to customer_source_table
INSERT INTO  customer_source_table
SELECT
  col1 AS userId,
  col2 AS name,
  col3 AS city
FROM (
  VALUES
  -- Initial load.
  (101, "Raul",     "Oaxaca"),
  (102, "Isabel",   "Monterrey"),
  (103, "Mercedes", "Tijuana"),
  (104, "Lily",     "Cancun")
);

num_affected_rows,num_inserted_rows
4,4


In [0]:
# running the streaming
calling_customer_cdf()

In [0]:
%sql
---Checking data ---
select * from customers_cdf

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,insert,1,2026-08-04T09:19:46.000Z
103,Mercedes,Tijuana,insert,1,2026-08-04T09:19:46.000Z
104,Lily,Cancun,insert,1,2026-08-04T09:19:46.000Z


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_history").display()

userId,name,city,__START_AT,__END_AT
101,Raul,Oaxaca,2026-08-04T09:19:46.000Z,null
102,Isabel,Monterrey,2026-08-04T09:19:46.000Z,null
103,Mercedes,Tijuana,2026-08-04T09:19:46.000Z,null
104,Lily,Cancun,2026-08-04T09:19:46.000Z,null


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_current").display()

userId,name,city
101,Raul,Oaxaca
102,Isabel,Monterrey
103,Mercedes,Tijuana
104,Lily,Cancun


In [0]:
%sql
insert into cdc_catalog.cdc_schema.users_cdf values (127, "Gunda",     "India",      "INSERT", 7)

In [0]:
spark.sql("select * from workspace.default.users_current").display()

In [0]:
%sql
insert into cdc_catalog.cdc_schema.users_cdf_source1 values(null, null,      null,          "TRUNCATE", 3)

In [0]:
spark.sql("select * from users_cdf").display()

In [0]:
spark.sql("select * from workspace.default.users_current").display()

In [0]:
%sql
select * from workspace.default.cdc_pipeline_type_1